# 01. Xử lý Dữ liệu — Tối ưu cho Kaggle (30GB RAM)

**Pipeline:**
1. JSONL → Parquet (chunked streaming)
2. Iterative K-Core filtering
3. Global Time-Split (chống data leakage)
4. Trích xuất text Train → `train_text_for_sentiment.parquet`
5. Lọc & xử lý Meta Data

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'orjson', 'polars', 'pyarrow'])

import os, gc, orjson, re
import pandas as pd
import numpy as np
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq

print("✅ Thư viện đã sẵn sàng!")

✅ Thư viện đã sẵn sàng!


In [2]:
# ============================================================
# CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ
# ============================================================
RAW_REVIEW_PATH = '/kaggle/input/datasets/shinnraa/data-amazon-raw/Clothing_Shoes_and_Jewelry.jsonl/Clothing_Shoes_and_Jewelry.jsonl'
RAW_META_PATH   = '/kaggle/input/datasets/shinnraa/data-amazon-raw/meta_Clothing_Shoes_and_Jewelry.jsonl/meta_Clothing_Shoes_and_Jewelry.jsonl'

WORKING_DIR = '/kaggle/working/processed'
os.makedirs(WORKING_DIR, exist_ok=True)

TMP_DIR = '/kaggle/working/tmp'
os.makedirs(TMP_DIR, exist_ok=True)

# File trung gian
INTERIM_RAW_PATH = os.path.join(TMP_DIR, 'interim_raw.parquet')

# File đầu ra
TRAIN_OUT_PATH        = os.path.join(WORKING_DIR, 'train_interactions.parquet')
TEST_OUT_PATH         = os.path.join(WORKING_DIR, 'test_interactions.parquet')
META_OUT_PATH         = os.path.join(WORKING_DIR, 'filtered_metadata.parquet')
MAPPING_OUT_PATH      = os.path.join(WORKING_DIR, 'item_mapping.parquet')
TRAIN_TEXT_OUT_PATH   = os.path.join(WORKING_DIR, 'train_text_for_sentiment.parquet')
USER_MAPPING_OUT_PATH = os.path.join(WORKING_DIR, 'user_mapping.parquet')

# Tham số
K_CORE      = 5
TRAIN_RATIO = 0.8
CHUNK_SIZE  = 2_000_000   # Giảm xuống 2M để an toàn RAM Kaggle

print("✅ Cấu hình hoàn tất!")
print(f"   WORKING_DIR : {WORKING_DIR}")
print(f"   K_CORE      : {K_CORE}")
print(f"   TRAIN_RATIO : {TRAIN_RATIO}")
print(f"   CHUNK_SIZE  : {CHUNK_SIZE:,}")

✅ Cấu hình hoàn tất!
   WORKING_DIR : /kaggle/working/processed
   K_CORE      : 5
   TRAIN_RATIO : 0.8
   CHUNK_SIZE  : 2,000,000


In [3]:
def get_fashion_asins(meta_path):
    print("BƯỚC 0: Quét Meta Data để lấy danh sách AMAZON_FASHION...")
    fashion_asins = set()
    total_scanned = 0
    
    with open(meta_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                data = orjson.loads(line)
                total_scanned += 1
                
                # Kiểm tra category
                cat = data.get('main_category')
                if cat and cat.strip().upper() == 'AMAZON FASHION': # Tùy format thực tế trong file
                    fashion_asins.add(data.get('parent_asin'))
            except Exception:
                continue

    print(f"✅ BƯỚC 0 HOÀN TẤT!")
    print(f"   Tổng số item đã quét: {total_scanned:,}")
    print(f"   Số item thuộc AMAZON FASHION: {len(fashion_asins):,}")
    return fashion_asins

# Thực thi lấy danh sách
FASHION_ASIN_SET = get_fashion_asins(RAW_META_PATH)

BƯỚC 0: Quét Meta Data để lấy danh sách AMAZON_FASHION...
✅ BƯỚC 0 HOÀN TẤT!
   Tổng số item đã quét: 7,218,481
   Số item thuộc AMAZON FASHION: 6,038,522


## Bước 1 — JSONL → Parquet (Streaming, 27GB)

In [4]:
def jsonl_to_parquet(input_path, output_path, chunk_size, valid_asin_set):
    print("BƯỚC 1: Chuyển đổi Review JSONL → Parquet (Đã lọc Fashion)...")
    writer = None
    buffer = []
    chunk_count = 0
    total_rows = 0

    with open(input_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                data = orjson.loads(line)
                asin = data.get('parent_asin')
                
                # CHẶN NGAY TẠI ĐÂY: Nếu không thuộc Fashion -> Bỏ qua luôn
                if asin not in valid_asin_set:
                    continue

                uid  = data.get('user_id')
                rat  = data.get('rating')
                ts   = data.get('timestamp')

                if uid is None or asin is None or ts is None:
                    continue

                buffer.append({
                    'user_id':     uid,
                    'parent_asin': asin,
                    'rating':      float(rat) if rat is not None else 0.0,
                    'timestamp':   int(ts)
                })

                if len(buffer) >= chunk_size:
                    chunk_count += 1
                    total_rows += len(buffer)
                    table = pa.Table.from_pandas(pd.DataFrame(buffer))
                    if writer is None:
                        writer = pq.ParquetWriter(output_path, table.schema, compression='zstd')
                    writer.write_table(table)
                    buffer.clear()
                    print(f"   ✔ Chunk {chunk_count} | Tổng {total_rows:,} dòng hợp lệ")
                    gc.collect()

            except Exception:
                continue

    if buffer:
        total_rows += len(buffer)
        table = pa.Table.from_pandas(pd.DataFrame(buffer))
        if writer is None:
            writer = pq.ParquetWriter(output_path, table.schema, compression='zstd')
        writer.write_table(table)

    if writer:
        writer.close()

    print(f"\n✅ BƯỚC 1 HOÀN TẤT — Tổng {total_rows:,} dòng thô thuộc AMAZON_FASHION.")
    return total_rows

# Gọi hàm với FASHION_ASIN_SET
total = jsonl_to_parquet(RAW_REVIEW_PATH, INTERIM_RAW_PATH, CHUNK_SIZE, FASHION_ASIN_SET)

BƯỚC 1: Chuyển đổi Review JSONL → Parquet (Đã lọc Fashion)...
   ✔ Chunk 1 | Tổng 2,000,000 dòng hợp lệ
   ✔ Chunk 2 | Tổng 4,000,000 dòng hợp lệ
   ✔ Chunk 3 | Tổng 6,000,000 dòng hợp lệ
   ✔ Chunk 4 | Tổng 8,000,000 dòng hợp lệ
   ✔ Chunk 5 | Tổng 10,000,000 dòng hợp lệ
   ✔ Chunk 6 | Tổng 12,000,000 dòng hợp lệ
   ✔ Chunk 7 | Tổng 14,000,000 dòng hợp lệ
   ✔ Chunk 8 | Tổng 16,000,000 dòng hợp lệ
   ✔ Chunk 9 | Tổng 18,000,000 dòng hợp lệ
   ✔ Chunk 10 | Tổng 20,000,000 dòng hợp lệ
   ✔ Chunk 11 | Tổng 22,000,000 dòng hợp lệ
   ✔ Chunk 12 | Tổng 24,000,000 dòng hợp lệ
   ✔ Chunk 13 | Tổng 26,000,000 dòng hợp lệ
   ✔ Chunk 14 | Tổng 28,000,000 dòng hợp lệ
   ✔ Chunk 15 | Tổng 30,000,000 dòng hợp lệ
   ✔ Chunk 16 | Tổng 32,000,000 dòng hợp lệ
   ✔ Chunk 17 | Tổng 34,000,000 dòng hợp lệ
   ✔ Chunk 18 | Tổng 36,000,000 dòng hợp lệ
   ✔ Chunk 19 | Tổng 38,000,000 dòng hợp lệ
   ✔ Chunk 20 | Tổng 40,000,000 dòng hợp lệ
   ✔ Chunk 21 | Tổng 42,000,000 dòng hợp lệ
   ✔ Chunk 22 | Tổng 44,000

## Bước 2 — Iterative K-Core Filtering

In [5]:
def apply_k_core_iterative(data_path, k=5, max_rounds=15):
    print(f"BƯỚC 2: Iterative K-Core (k={k})...")
    current_path = data_path

    for round_i in range(1, max_rounds + 1):
        # 1. ĐO SỐ LƯỢNG TRƯỚC KHI LỌC
        prev_count = pl.scan_parquet(current_path).select(pl.len()).collect().item()
        
        lf = pl.scan_parquet(current_path).select(['user_id', 'parent_asin'])

        valid_users = lf.group_by('user_id').len().filter(pl.col('len') >= k).select('user_id')
        valid_items = lf.group_by('parent_asin').len().filter(pl.col('len') >= k).select('parent_asin')

        filtered = (
            lf.join(valid_users, on='user_id', how='inner')
              .join(valid_items, on='parent_asin', how='inner')
              .unique()
        )

        checkpoint_path = os.path.join(TMP_DIR, f'kcore_round_{round_i}.parquet')
        filtered.sink_parquet(checkpoint_path)

        # 2. ĐO SỐ LƯỢNG SAU KHI LỌC
        curr_count = pl.scan_parquet(checkpoint_path).select(pl.len()).collect().item()
        print(f"   Vòng {round_i}: {prev_count:,} → {curr_count:,} tương tác")

        if current_path != data_path and os.path.exists(current_path):
            os.remove(current_path)

        current_path = checkpoint_path
        gc.collect()

        if curr_count == prev_count:
            print(f"   ✔ Hội tụ sau {round_i} vòng!")
            break

    valid_ids = pl.read_parquet(current_path)
    print(f"\n✅ BƯỚC 2 HOÀN TẤT — {curr_count:,} cặp (user, item) hợp lệ.")
    return valid_ids, current_path

valid_ids_df, kcore_final_path = apply_k_core_iterative(INTERIM_RAW_PATH, K_CORE)

BƯỚC 2: Iterative K-Core (k=5)...
   Vòng 1: 60,093,434 → 24,831,323 tương tác
   Vòng 2: 24,831,323 → 21,608,788 tương tác
   Vòng 3: 21,608,788 → 20,685,159 tương tác
   Vòng 4: 20,685,159 → 20,486,558 tương tác
   Vòng 5: 20,486,558 → 20,420,324 tương tác
   Vòng 6: 20,420,324 → 20,404,890 tương tác
   Vòng 7: 20,404,890 → 20,399,543 tương tác
   Vòng 8: 20,399,543 → 20,398,319 tương tác
   Vòng 9: 20,398,319 → 20,397,855 tương tác
   Vòng 10: 20,397,855 → 20,397,735 tương tác
   Vòng 11: 20,397,735 → 20,397,715 tương tác
   Vòng 12: 20,397,715 → 20,397,711 tương tác
   Vòng 13: 20,397,711 → 20,397,711 tương tác
   ✔ Hội tụ sau 13 vòng!

✅ BƯỚC 2 HOÀN TẤT — 20,397,711 cặp (user, item) hợp lệ.


## Bước 3 — Global Time-Split + ID Mapping (Chống Data Leakage)

In [6]:
import polars as pl
import os

def split_and_map_hybrid(raw_path, valid_ids, train_out, test_out):
    print("BƯỚC 3: Global Time-Split (80/20 -> Train/Test) & ID Mapping...")

    # ---------------------------------------------------------
    # 1. ID MAPPING
    # ---------------------------------------------------------
    valid_user_set = set(valid_ids['user_id'].to_list())
    valid_item_set = set(valid_ids['parent_asin'].to_list())

    u_map = pl.DataFrame({'user_id': sorted(valid_user_set)}).with_row_index('mapped_user_id', offset=1).with_columns(pl.col('mapped_user_id').cast(pl.Int32))
    i_map = pl.DataFrame({'parent_asin': sorted(valid_item_set)}).with_row_index('mapped_item_id', offset=1).with_columns(pl.col('mapped_item_id').cast(pl.Int32))

    lf = (
        pl.scan_parquet(raw_path)
          .join(u_map.lazy(), on='user_id', how='inner')
          .join(i_map.lazy(), on='parent_asin', how='inner')
          .select(['mapped_user_id', 'mapped_item_id', 'rating', 'timestamp'])
    )

    # ---------------------------------------------------------
    # 2. GLOBAL TIME-SPLIT (80/20) BẰNG QUANTILE (TỐI ƯU RAM)
    # ---------------------------------------------------------
    print("   Đang tìm mốc thời gian (Quantile 80%)...")
    cutoff_ts = lf.select(pl.col('timestamp').quantile(0.8, interpolation='nearest')).collect().item()
    print(f"   Mốc timestamp chia tập: {cutoff_ts}")

    # Chia Train 80% và Test 20%
    train_global_lf = lf.filter(pl.col('timestamp') <= cutoff_ts)
    test_global_lf  = lf.filter(pl.col('timestamp') > cutoff_ts)

    print("   Đang ghi tập Train gốc...")
    train_global_lf.sink_parquet(train_out)

    # ---------------------------------------------------------
    # 3. LỌC COLD-START CHO TẬP TEST GỐC
    # ---------------------------------------------------------
    print("   Đang lọc cold-start và ghi tập Test...")
    train_items = pl.scan_parquet(train_out).select('mapped_item_id').unique()

    (
        test_global_lf
        .join(train_items, on='mapped_item_id', how='inner')
        .sink_parquet(test_out)
    )

    # ---------------------------------------------------------
    # 4. THỐNG KÊ KẾT QUẢ
    # ---------------------------------------------------------
    train_size = pl.scan_parquet(train_out).select(pl.len()).collect().item()
    test_size  = pl.scan_parquet(test_out).select(pl.len()).collect().item()

    print(f"\n✅ BƯỚC 3 HOÀN TẤT!")
    print(f"   Train gốc (80%) : {train_size:,} dòng")
    print(f"   Test gốc  (20%) : {test_size:,} dòng")

    return valid_item_set, i_map, u_map

# --- Gọi hàm thực thi ---
# Ta xóa bỏ các tham số SASREC_TRAIN_PATH, SASREC_VAL_PATH ở đây
valid_items, item_map, user_map = split_and_map_hybrid(
    INTERIM_RAW_PATH, valid_ids_df, 
    TRAIN_OUT_PATH, TEST_OUT_PATH
)

# Lưu hai bảng Mapping ID
item_map.write_parquet(MAPPING_OUT_PATH)
user_map.write_parquet(USER_MAPPING_OUT_PATH)

BƯỚC 3: Global Time-Split (80/20 -> Train/Test) & ID Mapping...
   Đang tìm mốc thời gian (Quantile 80%)...
   Mốc timestamp chia tập: 1646401448362.0
   Đang ghi tập Train gốc...
   Đang lọc cold-start và ghi tập Test...

✅ BƯỚC 3 HOÀN TẤT!
   Train gốc (80%) : 16,509,306 dòng
   Test gốc  (20%) : 3,200,835 dòng


## Bước 4 — Trích xuất Text từ Train cho RoBERTa

> **Chỉ lấy các review nằm trong tập Train** để tránh data leakage khi huấn luyện sentiment model.

In [7]:
def extract_train_text_optimized_v2(raw_review_path, train_parquet_path, output_path, chunk_size=500_000):
    print("BƯỚC 4: Trích xuất text Train (Lưu trực tiếp Mapped ID & Timestamp)...")

    # 1. Load Mapping dictionaries để chuyển đổi từ String ID -> Integer ID (O(1))
    print("   Đang tạo từ điển mapping trong RAM...")
    user_map = pl.read_parquet(USER_MAPPING_OUT_PATH)
    item_map = pl.read_parquet(MAPPING_OUT_PATH)
    
    user_dict = dict(zip(user_map['user_id'].to_list(), user_map['mapped_user_id'].to_list()))
    item_dict = dict(zip(item_map['parent_asin'].to_list(), item_map['mapped_item_id'].to_list()))
    
    # Xoá dataframe mapping để giải phóng RAM
    del user_map, item_map
    gc.collect()

    # 2. Build Set O(1) trực tiếp từ tập Train (Dùng mapped_id và timestamp)
    print("   Đang build train set...")
    # Tập train vốn đã chứa sẵn 'mapped_user_id', 'mapped_item_id', 'timestamp' nên ta chỉ việc đọc lên
    train_df = pl.read_parquet(train_parquet_path, columns=['mapped_user_id', 'mapped_item_id', 'timestamp'])
    
    # Đưa vào tuple 3 giá trị để chống Data Leakage triệt để
    train_set = set(
        zip(train_df['mapped_user_id'].to_list(),
            train_df['mapped_item_id'].to_list(),
            train_df['timestamp'].to_list())
    )
    
    del train_df
    gc.collect()
    print(f"   Train set: {len(train_set):,} tương tác")

    # 3. Quét file JSONL gốc, tra cứu và trích xuất text
    writer = None
    buffer = []
    found   = 0
    skipped = 0

    print("   Đang quét JSONL và trích xuất text...")
    with open(raw_review_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                data  = orjson.loads(line)
                uid   = data.get('user_id')
                asin  = data.get('parent_asin')
                ts    = data.get('timestamp')

                if uid is None or asin is None or ts is None:
                    continue

                # Translate từ String ID sang Integer Mapped ID
                m_uid = user_dict.get(uid)
                m_iid = item_dict.get(asin) #user_id
                ts    = int(ts)

                # Điều kiện lọc:
                # Nếu ID không tồn tại trong map (tức là đã bị loại ở K-Core)
                # Hoặc tuple (user, item, timestamp) không có mặt trong tập Train gốc
                if m_uid is None or m_iid is None or (m_uid, m_iid, ts) not in train_set:
                    skipped += 1
                    continue

                title = str(data.get('title', '') or '').strip()
                text  = str(data.get('text',  '') or '').strip()
                combined = (title + ' ' + text).strip()
                
                # Nếu không có text gì cả thì bỏ qua cho đỡ tốn dung lượng
                if not combined:
                    combined = "no review" # Chuỗi an toàn cho mọi Tokenizer

                # LƯU TRỰC TIẾP MAPPED_ID
                buffer.append({
                    'mapped_user_id': m_uid,
                    'mapped_item_id': m_iid,
                    'rating':         float(data.get('rating', 0) or 0),
                    'text':           combined
                })
                found += 1

                # Phân trang ghi file Parquet
                if len(buffer) >= chunk_size:
                    table = pl.DataFrame(buffer).to_arrow()
                    if writer is None:
                        writer = pq.ParquetWriter(output_path, table.schema, compression='zstd')
                    writer.write_table(table)
                    buffer.clear()
                    print(f"   ✔ Đã ghi {found:,} dòng text...")
                    gc.collect()

            except Exception:
                continue

    # Ghi phần dữ liệu còn dư lại cuối cùng
    if buffer:
        table = pl.DataFrame(buffer).to_arrow()
        if writer is None:
            writer = pq.ParquetWriter(output_path, table.schema, compression='zstd')
        writer.write_table(table)

    if writer:
        writer.close()

    print(f"\n✅ BƯỚC 4 HOÀN TẤT")
    print(f"   Đã trích xuất : {found:,} dòng có text")
    print(f"   Bỏ qua        : {skipped:,} dòng")
    print(f"   Lưu tại       : {output_path}")

# --- Gọi Hàm ---
# Giả sử các biến path RAW_REVIEW_PATH, TRAIN_OUT_PATH,... đã được định nghĩa ở ô Cấu hình trên cùng
extract_train_text_optimized_v2(RAW_REVIEW_PATH, TRAIN_OUT_PATH, TRAIN_TEXT_OUT_PATH)

BƯỚC 4: Trích xuất text Train (Lưu trực tiếp Mapped ID & Timestamp)...
   Đang tạo từ điển mapping trong RAM...
   Đang build train set...
   Train set: 16,400,491 tương tác
   Đang quét JSONL và trích xuất text...
   ✔ Đã ghi 500,000 dòng text...
   ✔ Đã ghi 1,000,000 dòng text...
   ✔ Đã ghi 1,500,000 dòng text...
   ✔ Đã ghi 2,000,000 dòng text...
   ✔ Đã ghi 2,500,000 dòng text...
   ✔ Đã ghi 3,000,000 dòng text...
   ✔ Đã ghi 3,500,000 dòng text...
   ✔ Đã ghi 4,000,000 dòng text...
   ✔ Đã ghi 4,500,000 dòng text...
   ✔ Đã ghi 5,000,000 dòng text...
   ✔ Đã ghi 5,500,000 dòng text...
   ✔ Đã ghi 6,000,000 dòng text...
   ✔ Đã ghi 6,500,000 dòng text...
   ✔ Đã ghi 7,000,000 dòng text...
   ✔ Đã ghi 7,500,000 dòng text...
   ✔ Đã ghi 8,000,000 dòng text...
   ✔ Đã ghi 8,500,000 dòng text...
   ✔ Đã ghi 9,000,000 dòng text...
   ✔ Đã ghi 9,500,000 dòng text...
   ✔ Đã ghi 10,000,000 dòng text...
   ✔ Đã ghi 10,500,000 dòng text...
   ✔ Đã ghi 11,000,000 dòng text...
   ✔ Đã ghi 11

## Bước 5 — Lọc & Xử lý Meta Data (17GB)

In [8]:
def process_meta(meta_path, valid_items_set, item_map_df, output_path, chunk_size):
    print("BƯỚC 5: Lọc và Xử lý Meta Data (Chống Leakage)...")

    def clean_number(val):
        try:
            if isinstance(val, str):
                nums = re.findall(r'\d+\.?\d*', val)
                return float(nums[0]) if nums else None
            return float(val) if val is not None else None
        except Exception:
            return None

    tmp_path = output_path + '.tmp.parquet'
    writer   = None
    buffer   = []
    found    = 0

    with open(meta_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                data = orjson.loads(line)
                asin = data.get('parent_asin')

                if asin not in valid_items_set:
                    continue

                buffer.append({
                    'parent_asin':    asin,
                    'price':          clean_number(data.get('price')),
                    'store':          str(data.get('store', 'Unknown') or 'Unknown')
                    
                })
                found += 1

                if len(buffer) >= chunk_size:
                    table = pa.Table.from_pandas(pd.DataFrame(buffer))
                    if writer is None:
                        writer = pq.ParquetWriter(tmp_path, table.schema, compression='zstd')
                    writer.write_table(table)
                    buffer.clear()
                    print(f"   ✔ Đã ghi {found:,} item meta...")
                    gc.collect()

            except Exception:
                continue

    if buffer:
        table = pa.Table.from_pandas(pd.DataFrame(buffer))
        if writer is None:
            writer = pq.ParquetWriter(tmp_path, table.schema, compression='zstd')
        writer.write_table(table)

    if writer:
        writer.close()

    print("   Đang gắn mapped_item_id và điền Trung vị (Median) cho Price...")
    median_price = (pl.scan_parquet(tmp_path)
                        .select(pl.col('price').median())
                        .collect().item())
    (
        pl.scan_parquet(tmp_path)
          .join(item_map_df.lazy(), on='parent_asin', how='inner')
          .with_columns([
              pl.col('price').fill_null(median_price)
          ])
          .drop('parent_asin')
          .sink_parquet(output_path)
    )
    os.remove(tmp_path)

    print(f"\n✅ BƯỚC 5 HOÀN TẤT — {found:,} item meta đã xử lý an toàn")
    print(f"   Lưu tại: {output_path}")

process_meta(RAW_META_PATH, valid_items, item_map, META_OUT_PATH, CHUNK_SIZE)

BƯỚC 5: Lọc và Xử lý Meta Data (Chống Leakage)...
   Đang gắn mapped_item_id và điền Trung vị (Median) cho Price...

✅ BƯỚC 5 HOÀN TẤT — 626,748 item meta đã xử lý an toàn
   Lưu tại: /kaggle/working/processed/filtered_metadata.parquet


In [9]:
import polars as pl
import os

def compute_item_statistics_from_train(train_path, meta_path, output_meta_path):
    print("BƯỚC 6: Tính toán lại average_rating và rating_number từ tập Train...")
    
    # 1. Tính toán thống kê từ tập Train
    # train_path chính là 'train_interactions.parquet'
    train_stats = (
        pl.scan_parquet(train_path)
        .group_by('mapped_item_id')
        .agg([
            pl.col('rating').mean().alias('train_average_rating'), # Tính trung bình rating
            pl.col('rating').count().alias('train_rating_number')  # Đếm số lượng rating
        ])
    )
    
    # 2. Đọc file Metadata đã xử lý ở Bước 5
    # meta_path chính là 'filtered_metadata.parquet'
    # Nối (Join) các thống kê vừa tính được vào file Meta
    enriched_meta = (
        pl.scan_parquet(meta_path)
        .join(train_stats, on='mapped_item_id', how='left') # Left join để giữ toàn bộ item trong Meta
        .with_columns([
            # Nếu có item nào trong meta mà chưa có rating trong train (hiếm gặp nếu bạn đã lọc k-core tốt), 
            # có thể fill_null bằng 0 hoặc 1 giá trị mặc định.
            pl.col('train_average_rating').fill_null(0.0),
            pl.col('train_rating_number').fill_null(0).cast(pl.Int32)
        ])
        .collect()
    )
    
    # 3. Ghi đè lại hoặc tạo file mới
    enriched_meta.write_parquet(output_meta_path)
    
    print(f"✅ Đã tính xong! File Metadata mới có thêm cột 'train_average_rating' và 'train_rating_number'.")
    print(f"   Lưu tại: {output_meta_path}")

# --- CÁCH GỌI HÀM ---
# Giả sử bạn đã chạy xong Bước 5 và có file META_OUT_PATH
compute_item_statistics_from_train(TRAIN_OUT_PATH, META_OUT_PATH, META_OUT_PATH)

BƯỚC 6: Tính toán lại average_rating và rating_number từ tập Train...
✅ Đã tính xong! File Metadata mới có thêm cột 'train_average_rating' và 'train_rating_number'.
   Lưu tại: /kaggle/working/processed/filtered_metadata.parquet


## Dọn dẹp & Tổng kết

In [10]:
# Xoá file trung gian
for tmp_file in [INTERIM_RAW_PATH]:
    if os.path.exists(tmp_file):
        os.remove(tmp_file)
        print(f"   Đã xoá tạm: {tmp_file}")

# Xoá checkpoint K-Core
import glob
for f in glob.glob(os.path.join(TMP_DIR, 'kcore_round_*.parquet')):
    os.remove(f)

print("\n" + "="*55)
print("  HOÀN TẤT TOÀN BỘ FILE 01!")
print("="*55)
print("\nCác file đầu ra:")
for path in [
    TRAIN_OUT_PATH, TEST_OUT_PATH,
    META_OUT_PATH, MAPPING_OUT_PATH,
    TRAIN_TEXT_OUT_PATH
    
]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024**2
        print(f"  ✔ {os.path.basename(path):45s} {size_mb:8.1f} MB")
    else:
        print(f"  ✘ {os.path.basename(path):45s} KHÔNG TỒN TẠI")

   Đã xoá tạm: /kaggle/working/tmp/interim_raw.parquet

  HOÀN TẤT TOÀN BỘ FILE 01!

Các file đầu ra:
  ✔ train_interactions.parquet                       174.4 MB
  ✔ test_interactions.parquet                         37.1 MB
  ✔ filtered_metadata.parquet                          6.2 MB
  ✔ item_mapping.parquet                               4.6 MB
  ✔ train_text_for_sentiment.parquet                1215.6 MB
